In [1]:
from pathlib import Path
import time
import pandas as pd
import numpy as np

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

files = sorted(DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet"))

len(files), [f.name for f in files]

(12,
 ['yellow_tripdata_2025-01_clean.parquet',
  'yellow_tripdata_2025-02_clean.parquet',
  'yellow_tripdata_2025-03_clean.parquet',
  'yellow_tripdata_2025-04_clean.parquet',
  'yellow_tripdata_2025-05_clean.parquet',
  'yellow_tripdata_2025-06_clean.parquet',
  'yellow_tripdata_2025-07_clean.parquet',
  'yellow_tripdata_2025-08_clean.parquet',
  'yellow_tripdata_2025-09_clean.parquet',
  'yellow_tripdata_2025-10_clean.parquet',
  'yellow_tripdata_2025-11_clean.parquet',
  'yellow_tripdata_2025-12_clean.parquet'])

In [2]:
tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

for name, tier_files in tiers.items():
    print(name, len(tier_files))

1_month 1
3_months 3
6_months 6
12_months 12


In [3]:
# timing helper
def timed(func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    return result, elapsed

In [9]:
def benchmark_pandas(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed(
        lambda: pd.concat(
            [pd.read_parquet(f) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null handling
    null_counts, results["null_count"] = timed(
        lambda: df.isna().sum()
    )

    # 4. Groupby aggregation
    grouped, results["groupby"] = timed(
        lambda: df.groupby("payment_type").agg(
            trip_count=("VendorID", "size"),
            avg_distance=("trip_distance", "mean"),
            avg_fare=("fare_amount", "mean"),
            avg_tip=("tip_amount", "mean"),
            avg_total=("total_amount", "mean"),
        )
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        temp["tip_pct"] = np.where(
            temp["fare_amount"] > 0,
            temp["tip_amount"] / temp["fare_amount"] * 100,
            np.nan
        )

        return temp

    engineered, results["feature_engineering"] = timed(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in [
            "load",
            "filter",
            "null_count",
            "groupby",
            "feature_engineering",
            "sort",
        ]
    )

    return results

In [10]:
test_results = benchmark_pandas(tiers["1_month"])
test_results

{'load': 0.05625303198758047,
 'filter': 0.0815756079973653,
 'null_count': 0.0418932769971434,
 'groupby': 0.08085664598911535,
 'feature_engineering': 0.08384074800414965,
 'sort': 0.5546514769957867,
 'rows': 3475082,
 'total': 0.8990707879711408}

In [11]:
pandas_results = []

for tier_name, tier_files in tiers.items():
    print(f"Running pandas benchmark: {tier_name}")

    result = benchmark_pandas(tier_files)

    result["tier"] = tier_name
    result["backend"] = "pandas"

    pandas_results.append(result)

    print(result)

Running pandas benchmark: 1_month
{'load': 0.05773666698951274, 'filter': 0.08839530000113882, 'null_count': 0.04274086300574709, 'groupby': 0.08101859899761621, 'feature_engineering': 0.08410511599504389, 'sort': 0.5564500720065553, 'rows': 3475082, 'total': 0.910446616995614, 'tier': '1_month', 'backend': 'pandas'}
Running pandas benchmark: 3_months
{'load': 0.23267035200842656, 'filter': 0.2673865120013943, 'null_count': 0.13441045800573193, 'groupby': 0.259550577000482, 'feature_engineering': 0.4070900679944316, 'sort': 2.4316642229969148, 'rows': 11197681, 'total': 3.732772190007381, 'tier': '3_months', 'backend': 'pandas'}
Running pandas benchmark: 6_months
{'load': 0.4763363560050493, 'filter': 0.5734818159980932, 'null_count': 0.31247629200515803, 'groupby': 0.5528942579985596, 'feature_engineering': 0.8796049120137468, 'sort': 5.700846114006708, 'rows': 24082473, 'total': 8.495639748027315, 'tier': '6_months', 'backend': 'pandas'}
Running pandas benchmark: 12_months
{'load': 0

In [12]:
pandas_results_df = pd.DataFrame(pandas_results)

pandas_results_df

,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend
0,0.057737,0.088395,0.042741,0.081019,0.084105,0.556450,3475082,0.910447,1_month,pandas
1,0.232670,0.267387,0.134410,0.259551,0.407090,2.431664,11197681,3.732772,3_months,pandas
2,0.476336,0.573482,0.312476,0.552894,0.879605,5.700846,24082473,8.495640,6_months,pandas
3,0.987617,1.196741,0.584456,1.117304,1.835112,12.197595,48720015,17.918826,12_months,pandas


In [14]:
#cpu benchmark notes
#predictable linear scaling of time with data size, as expected. The groupby and feature engineering steps are the most time-consuming, which is consistent with the complexity of these operations.

In [15]:
#switching to GPU backend for benchmarking. The GPU backend is expected to provide significant speedup for large datasets, especially for operations that can be parallelized effectively.

In [1]:
from pathlib import Path
import time

import cudf
import cupy as cp

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

files = sorted(
    DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet")
)

tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

print("Files:", len(files))

Files: 12


In [2]:
def timed_gpu(func):
    cp.cuda.Stream.null.synchronize()

    start = time.perf_counter()

    result = func()

    cp.cuda.Stream.null.synchronize()

    elapsed = time.perf_counter() - start

    return result, elapsed

In [3]:
def benchmark_cudf(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed_gpu(
        lambda: cudf.concat(
            [cudf.read_parquet(str(f)) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed_gpu(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null handling
    null_counts, results["null_count"] = timed_gpu(
        lambda: df.isna().sum()
    )

    # 4. Groupby aggregation
    grouped, results["groupby"] = timed_gpu(
        lambda: df.groupby("payment_type").agg({
            "VendorID": "count",
            "trip_distance": "mean",
            "fare_amount": "mean",
            "tip_amount": "mean",
            "total_amount": "mean",
        })
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        tip_pct = (
            temp["tip_amount"]
            / temp["fare_amount"]
            * 100
        )

        temp["tip_pct"] = tip_pct.where(
            temp["fare_amount"] > 0
        )

        return temp

    engineered, results["feature_engineering"] = timed_gpu(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed_gpu(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in [
            "load",
            "filter",
            "null_count",
            "groupby",
            "feature_engineering",
            "sort",
        ]
    )

    return results

In [4]:
test_gpu = benchmark_cudf(tiers["1_month"])

test_gpu

{'load': 0.7353032739920309,
 'filter': 0.07126734900521114,
 'null_count': 0.06203560400172137,
 'groupby': 0.042866095987847075,
 'feature_engineering': 4.382151893005357,
 'sort': 0.07480390000273474,
 'rows': 3475082,
 'total': 5.368428115994902}

In [5]:
test_gpu["rows"]

3475082

In [6]:
cudf_results = []

for tier_name, tier_files in tiers.items():

    print(f"Running cuDF benchmark: {tier_name}")

    result = benchmark_cudf(tier_files)

    result["tier"] = tier_name
    result["backend"] = "cuDF"

    cudf_results.append(result)

    print(result)

Running cuDF benchmark: 1_month
{'load': 0.11933359800605103, 'filter': 0.03420211200136691, 'null_count': 0.021342646999983117, 'groupby': 0.007892302994150668, 'feature_engineering': 0.0581556449906202, 'sort': 0.06224926799768582, 'rows': 3475082, 'total': 0.30317557298985776, 'tier': '1_month', 'backend': 'cuDF'}
Running cuDF benchmark: 3_months
{'load': 0.2679533960035769, 'filter': 0.09664664400042966, 'null_count': 0.045506852999096736, 'groupby': 0.021655976001056843, 'feature_engineering': 0.1554580500087468, 'sort': 0.20936744600476231, 'rows': 11197681, 'total': 0.7965883650176693, 'tier': '3_months', 'backend': 'cuDF'}
Running cuDF benchmark: 6_months
{'load': 0.5341708680061856, 'filter': 0.1964125100057572, 'null_count': 0.07760185599909164, 'groupby': 0.04437339499418158, 'feature_engineering': 0.3160693889949471, 'sort': 0.45486281599733047, 'rows': 24082473, 'total': 1.6234908339974936, 'tier': '6_months', 'backend': 'cuDF'}
Running cuDF benchmark: 12_months
{'load': 1

In [7]:
cudf_results_df = cudf.DataFrame(cudf_results)

cudf_results_df

,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend
0,0.119334,0.034202,0.021343,0.007892,0.058156,0.062249,3475082,0.303176,1_month,cuDF
1,0.267953,0.096647,0.045507,0.021656,0.155458,0.209367,11197681,0.796588,3_months,cuDF
2,0.534171,0.196413,0.077602,0.044373,0.316069,0.454863,24082473,1.623491,6_months,cuDF
3,1.068355,0.386197,0.135171,0.090657,2.171076,0.935869,48720015,4.787324,12_months,cuDF


In [8]:
#loading in pandas is faster than loading in cuDF, which is expected due to the overhead of transferring data to the GPU. However, for larger datasets, the GPU backend is expected to outperform pandas in subsequent operations.
#sorting is also faster in cuDF, which is expected due to the parallelization capabilities of GPUs. Overall, the GPU backend shows significant performance improvements for larger datasets, especially in operations that can be parallelized effectively.
# note, over 48.7million rows, pandas ~17.9 seconds, cuDF ~4.8 seconds
# will repeat 5x and report average times for each operation to get a more robust comparison between the two backends.